## Example 1: Basic article extraction

In [ ]:
from readabilipy import simple_json_from_html_string

# Example 1: Basic article extraction
print("Example 1: Basic article extraction")

try:
    # A small page: a header, the article itself, a sidebar and a footer
    sample_html = """
    <html>
        <head><title>Test Article</title></head>
        <body>
            <div id="header">Navigation menu</div>
            <article>
                <h1>Climate Change and Its Effects</h1>
                <p>Climate change refers to long-term shifts in global temperatures and weather patterns. 
                While climate change is natural, human activities have been the main driver since the 1800s.</p>
                <p>The primary cause of climate change is burning fossil fuels like coal, oil and gas. 
                This releases greenhouse gases into the atmosphere.</p>
                <p>The consequences of climate change include rising sea levels, extreme weather events, 
                and threats to biodiversity and food security.</p>
            </article>
            <div id="sidebar">Advertisement content</div>
            <footer>Copyright notice</footer>
        </body>
    </html>
    """

    # use_readability=False selects readabilipy's pure-Python extractor (the default, True, needs Node.js)
    article = simple_json_from_html_string(sample_html, use_readability=False)
    print(f"Result keys: {sorted(article)}")
    assert set(article) == {'title', 'byline', 'date', 'content', 'plain_content', 'plain_text'}, sorted(article)

    texts = [block['text'] for block in article['plain_text']]
    print(f"Title: {article['title']!r}")
    for text in texts:
        print(f"  - {text[:90]}")

    assert article['title'] == 'Test Article', article['title']
    assert article['byline'] is None and article['date'] is None, article  # no author or date metadata in the page

    # The heading and the three paragraphs come through in document order, with the line breaks collapsed
    expected = [
        'Climate Change and Its Effects',
        'Climate change refers to long-term shifts in global temperatures and weather patterns. '
        'While climate change is natural, human activities have been the main driver since the 1800s.',
        'The primary cause of climate change is burning fossil fuels like coal, oil and gas. '
        'This releases greenhouse gases into the atmosphere.',
        'The consequences of climate change include rising sea levels, extreme weather events, '
        'and threats to biodiversity and food security.',
    ]
    positions = [texts.index(text) for text in expected]
    assert positions == sorted(positions), positions

    # content is simplified HTML in a single <div> that keeps the article markup
    assert article['content'].startswith('<div>') and article['content'].endswith('</div>'), article['content'][:60]
    assert '<h1>Climate Change and Its Effects</h1>' in article['content'], article['content']
    # The pure-Python extractor does not score boilerplate blocks away: header, sidebar and footer text stay
    assert {'Navigation menu', 'Advertisement content', 'Copyright notice'} <= set(texts), texts

    # Simple readability figures from the extracted article text
    body = expected[1:]
    word_count = sum(len(text.split()) for text in body)
    sentence_count = sum(text.count('. ') + text.endswith('.') for text in body)
    print(f"Article paragraphs: {len(body)}, words: {word_count}, sentences: {sentence_count}")
    assert (word_count, sentence_count) == (68, 5), (word_count, sentence_count)

    print("Basic article extraction test completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"readabilipy example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Extraction configuration

In [ ]:
from readabilipy import simple_json_from_html_string, simple_tree_from_html_string

# Example 2: Extraction configuration
print("Example 2: Extraction configuration")

try:
    complex_html = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Advanced Web Article</title>
        <meta name="description" content="Article description">
        <meta name="author" content="John Doe">
    </head>
    <body>
        <nav>
            <ul><li><a href="/home">Home</a></li><li><a href="/about">About</a></li></ul>
        </nav>
        <main>
            <h1>Artificial Intelligence in Healthcare</h1>
            <div class="article-meta">
                <span class="author">Dr. Jane Smith</span>
                <span class="date">2023-12-01</span>
            </div>
            <div class="content">
                <p class="lead">Artificial intelligence is transforming healthcare through innovative applications 
                that improve patient care, reduce costs, and enhance diagnostic accuracy.</p>
                
                <h2>Machine Learning in Diagnosis</h2>
                <p>Machine learning algorithms can analyze medical images with remarkable precision. 
                Studies show AI can detect certain conditions earlier than traditional methods.</p>
                
                <h2>Natural Language Processing</h2>
                <p>NLP helps process clinical notes, extracting valuable insights from unstructured data. 
                This technology streamlines documentation and improves care coordination.</p>
                
                <blockquote>
                    "AI will augment human capabilities rather than replace healthcare professionals."
                    - Medical AI Expert
                </blockquote>
            </div>
        </main>
        <aside>
            <div class="ads">Advertisement</div>
            <div class="social">Share buttons</div>
        </aside>
        <footer>
            <p>© 2023 Medical Journal</p>
        </footer>
    </body>
    </html>
    """

    print(f"HTML length: {len(complex_html)} characters")
    default = simple_json_from_html_string(complex_html, use_readability=False)
    tagged = simple_json_from_html_string(complex_html, use_readability=False, content_digests=True, node_indexes=True)
    tree = simple_tree_from_html_string(complex_html)

    # simple_tree_from_html_string returns the same simplified HTML as a BeautifulSoup element
    print(f"Tree: {type(tree).__name__} <{tree.name}>, {len(str(tree))} characters")
    assert tree.name == 'div' and str(tree) == default['content'], str(tree)[:80]

    # content_digests and node_indexes only annotate plain_content; content is identical either way
    print(f"Annotated plain_content: {tagged['plain_content'][:120]}...")
    assert tagged['content'] == default['content']
    assert 'data-content-digest=' in tagged['plain_content'] and 'data-node-index=' in tagged['plain_content']
    assert 'data-' not in default['plain_content']

    # The simplifier drops <nav>, <head> metadata and every attribute, and keeps the structural tags
    content = default['content']
    for dropped in ('Home', 'About', 'href=', 'class=', 'lang=', 'Article description', 'charset'):
        assert dropped not in content, dropped
    for kept in ('<h1>Artificial Intelligence in Healthcare</h1>', '<h2>Machine Learning in Diagnosis</h2>',
                 '<h2>Natural Language Processing</h2>', '<blockquote>'):
        assert kept in content, kept
    print(f"Simplified content: {len(content)} characters (from {len(complex_html)})")

    # Metadata: the title comes from <title>; <meta name="author"> is not read as a byline on this path
    print(f"Title: {default['title']!r}, byline: {default['byline']!r}, date: {default['date']!r}")
    assert (default['title'], default['byline'], default['date']) == ('Advanced Web Article', None, None), default

    texts = [block['text'] for block in default['plain_text']]
    assert 'Dr. Jane Smith 2023-12-01' in texts, texts
    assert '"AI will augment human capabilities rather than replace healthcare professionals." - Medical AI Expert' in texts, texts

    print("Extraction configuration test completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"readabilipy example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Content filtering and quality assessment

In [ ]:
from readabilipy import simple_json_from_html_string
import hashlib

# Example 3: Content filtering and quality assessment, on readabilipy's own extraction
print("Example 3: Content filtering and quality assessment")

try:
    # Test content filtering capabilities
    test_content_samples = [
        {
            'name': 'High Quality Article',
            'html': '''<article>
                <h1>Climate Change Solutions</h1>
                <p>Renewable energy technologies are becoming increasingly cost-effective. Solar and wind power now compete with fossil fuels in many markets.</p>
                <p>Government policies play a crucial role in accelerating the transition to clean energy. Carbon pricing and renewable energy standards drive investment.</p>
                <p>Individual actions also matter. Energy efficiency improvements and sustainable transportation choices contribute to emission reductions.</p>
            </article>''',
            'expected_paragraphs': 4, 'expected_words': 58, 'expected_quality': 'high'
        },
        {
            'name': 'Short Article',
            'html': '''<div>
                <h2>Quick Update</h2>
                <p>New study published today.</p>
            </div>''',
            'expected_paragraphs': 2, 'expected_words': 6, 'expected_quality': 'low'
        },
        {
            'name': 'Navigation Heavy Page',
            'html': '''<body>
                <nav><ul><li>Link1</li><li>Link2</li><li>Link3</li></ul></nav>
                <div class="sidebar"><ul><li>Ad1</li><li>Ad2</li></ul></div>
                <p>Small content piece here.</p>
                <footer><div>Footer links</div></footer>
            </body>''',
            'expected_paragraphs': 3, 'expected_words': 10, 'expected_quality': 'low'
        },
        {
            'name': 'Well Structured Article',
            'html': '''<main>
                <h1>Machine Learning Fundamentals</h1>
                <section>
                    <h2>Supervised Learning</h2>
                    <p>Supervised learning uses labeled training data to learn a mapping from inputs to outputs. Common algorithms include linear regression and decision trees.</p>
                </section>
                <section>
                    <h2>Unsupervised Learning</h2>
                    <p>Unsupervised learning finds patterns in data without labels. Clustering and dimensionality reduction are typical applications.</p>
                </section>
            </main>''',
            'expected_paragraphs': 5, 'expected_words': 44, 'expected_quality': 'high'
        }
    ]
    
    print(f"Testing {len(test_content_samples)} content samples:")
    
    extracted = {}
    for sample in test_content_samples:
        print(f"\n--- {sample['name']} ---")
        result = simple_json_from_html_string(sample['html'], use_readability=False)
        assert set(result) == {'title', 'byline', 'date', 'content', 'plain_content', 'plain_text'}, sorted(result)
        paragraphs = [block['text'] for block in result['plain_text']]
        word_count = sum(len(text.split()) for text in paragraphs)
        # Quality from the words readabilipy kept, not from the raw markup
        quality_score = 'high' if word_count >= 40 else 'medium' if word_count >= 20 else 'low'
        print(f"  Paragraphs: {len(paragraphs)}, first: {paragraphs[0]!r}")
        print(f"  Word count: {word_count}")
        print(f"  Quality assessment: {quality_score} (expected {sample['expected_quality']})")
        assert not any('<' in text for text in paragraphs), paragraphs
        assert (len(paragraphs), word_count, quality_score) == (
            sample['expected_paragraphs'], sample['expected_words'], sample['expected_quality']), (sample['name'], paragraphs)
        extracted[sample['name']] = result
    
    # Content filtering: the <nav> is dropped; the pure-Python path keeps the sidebar list and the footer (measured)
    print(f"\nContent filtering (Navigation Heavy Page):")
    nav_html = test_content_samples[2]['html']
    nav_result = extracted['Navigation Heavy Page']
    nav_paragraphs = [block['text'] for block in nav_result['plain_text']]
    reduction = (1 - len(nav_result['content']) / len(nav_html)) * 100
    print(f"  Size reduction: {reduction:.1f}% ({len(nav_html)} -> {len(nav_result['content'])} chars)")
    print(f"  Kept: {nav_paragraphs}")
    assert 'Link1' not in nav_result['content'], nav_result['content']
    assert nav_paragraphs == ['* Ad1, * Ad2,', 'Small content piece here.', 'Footer links'], nav_paragraphs
    
    # Extraction options: node indexes and SHA-256 digests of each element's text go on plain_content, not content
    print(f"\nExtraction options (node_indexes, content_digests):")
    tagged = simple_json_from_html_string(nav_html, content_digests=True, node_indexes=True, use_readability=False)
    node_indexes = [block['node_index'] for block in tagged['plain_text']]
    digest = hashlib.sha256('Small content piece here.'.encode('utf-8')).hexdigest()
    print(f"  plain_text node indexes: {node_indexes}")
    print(f"  Paragraph digest: {digest[:16]}...")
    assert node_indexes == ['0.1.1', '0.2', '0.3.1'], tagged['plain_text']
    assert f'data-content-digest="{digest}"' in tagged['plain_content'], tagged['plain_content']
    assert 'data-node-index="0.2"' in tagged['plain_content'] and 'data-' not in tagged['content'], tagged
    
    # A full document: metadata extraction and text normalisation
    print(f"\nFull document metadata and normalisation:")
    article_html = """<html><head><title>Solar Power in 2024 | Example News</title>
<meta name="author" content="Jane Doe">
<meta property="article:published_time" content="2024-03-01T10:00:00Z">
<script>var tracking = 1;</script><style>p { color: red; }</style></head>
<body><nav><a href="/">Home</a> <a href="/news">News</a></nav>
<article><h1>Solar Power in 2024</h1>
<p>Costs  fell   while   efficiency rose.<!-- editor note --></p>
<p>Ceci est un article français. Dies ist ein Artikel über Technologie. Este es un artículo español.</p>
<p>The ﬁnal ﬁgure: ５０ GW.</p>
</article></body></html>"""
    article = simple_json_from_html_string(article_html, use_readability=False)
    texts = [block['text'] for block in article['plain_text']]
    print(f"  Title: {article['title']!r}, date: {article['date']!r}, byline: {article['byline']!r}")
    for text in texts:
        print(f"  - {text}")
    # the byline is extracted only on the Readability.js path (measured None here); the date loses its 'Z'
    assert (article['title'], article['date'], article['byline']) == (
        'Solar Power in 2024 | Example News', '2024-03-01T10:00:00', None), article
    # script, style, comments and <nav> are dropped; whitespace collapses; NFKC folds the ligature and full-width
    # digits; accented text is kept as written
    assert not any(s in article['content'] for s in ('tracking', 'color: red', 'editor note', 'Home')), article['content']
    for expected_text in ('Costs fell while efficiency rose.', 'The final figure: 50 GW.',
                          'Ceci est un article français. Dies ist ein Artikel über Technologie. Este es un artículo español.'):
        assert expected_text in texts, (expected_text, texts)
    
    print("Content filtering and quality assessment test completed")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"readabilipy example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Batch processing and output formats

In [ ]:
from readabilipy import simple_json_from_html_string
import csv
import json
import os
import tempfile

# Example 4: Batch processing and output formats
print("Example 4: Batch processing and output formats")

try:
    articles = [
        {
            'id': 'tech_001',
            'title': 'Future of Quantum Computing',
            'html': '''<article>
                <h1>Future of Quantum Computing</h1>
                <p>Quantum computing promises to revolutionize computational capabilities through quantum mechanical phenomena.</p>
                <p>Major tech companies are investing billions in quantum research and development.</p>
                <p>Applications range from cryptography to drug discovery and optimization problems.</p>
            </article>''',
            'source_url': 'https://tech.example.com/quantum-computing'
        },
        {
            'id': 'health_002', 
            'title': 'Personalized Medicine Advances',
            'html': '''<main>
                <h1>Personalized Medicine Advances</h1>
                <p>Genetic sequencing technologies are enabling precision treatments tailored to individual patients.</p>
                <p>Pharmacogenomics helps predict drug responses based on genetic profiles.</p>
                <p>AI algorithms analyze patient data to recommend personalized treatment plans.</p>
            </main>''',
            'source_url': 'https://health.example.com/personalized-medicine'
        },
        {
            'id': 'env_003',
            'title': 'Sustainable Agriculture Innovations',
            'html': '''<div class="content">
                <h1>Sustainable Agriculture Innovations</h1>
                <p>Vertical farming and hydroponics are reducing agricultural land use and water consumption.</p>
                <p>Precision agriculture uses sensors and AI to optimize crop yields while minimizing environmental impact.</p>
                <p>Gene editing technologies are developing drought-resistant and nutrient-rich crop varieties.</p>
            </div>''',
            'source_url': 'https://enviro.example.com/agriculture'
        }
    ]

    print(f"Processing {len(articles)} articles in batch:")
    results = []
    for article in articles:
        extracted = simple_json_from_html_string(article['html'], use_readability=False)
        texts = [block['text'] for block in extracted['plain_text']]
        record = {
            'id': article['id'],
            'source_url': article['source_url'],
            'heading': texts[0],
            'paragraphs': texts[1:],
            'word_count': sum(len(text.split()) for text in texts[1:]),
        }
        results.append(record)
        print(f"  {record['id']}: {record['heading']!r}, {len(record['paragraphs'])} paragraphs, {record['word_count']} words")
        # These fragments have no <title>, so title is None and the <h1> is the first text block
        assert extracted['title'] is None, extracted['title']
        assert record['heading'] == article['title'] and len(record['paragraphs']) == 3, record

    assert [r['word_count'] for r in results] == [32, 30, 36], [r['word_count'] for r in results]

    with tempfile.TemporaryDirectory() as temp_dir:
        # JSON: the full records round-trip unchanged
        json_path = os.path.join(temp_dir, 'articles.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        with open(json_path, encoding='utf-8') as f:
            assert json.load(f) == results

        # CSV: one summary row per article
        csv_path = os.path.join(temp_dir, 'articles.csv')
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['id', 'heading', 'word_count'])
            writer.writeheader()
            for r in results:
                writer.writerow({'id': r['id'], 'heading': r['heading'], 'word_count': r['word_count']})
        with open(csv_path, newline='', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
        assert [(row['id'], row['heading'], int(row['word_count'])) for row in rows] == [
            (r['id'], r['heading'], r['word_count']) for r in results], rows

        # Plain text: heading, blank line, paragraphs
        txt_path = os.path.join(temp_dir, 'articles.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            for r in results:
                f.write(r['heading'] + '\n\n' + '\n'.join(r['paragraphs']) + '\n\n')
        with open(txt_path, encoding='utf-8') as f:
            assert f.read().count('\n\n') == 2 * len(results)

        for path in (json_path, csv_path, txt_path):
            print(f"  {os.path.basename(path)}: {os.path.getsize(path)} bytes")

    # Filter and sort the extracted records
    longest_first = sorted(results, key=lambda r: r['word_count'], reverse=True)
    print(f"Longest first: {[r['id'] for r in longest_first]}")
    assert [r['id'] for r in longest_first] == ['env_003', 'tech_001', 'health_002']
    assert [r['id'] for r in results if r['word_count'] >= 32] == ['tech_001', 'env_003']

    print("Batch processing and output formats test completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"readabilipy example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Large, malformed and empty input

In [ ]:
from readabilipy import simple_json_from_html_string
import time

# Example 5: Large, malformed and empty input
print("Example 5: Large, malformed and empty input")

try:
    # A long page: 200 paragraphs plus script, style and navigation noise
    paragraphs = [f'Paragraph number {i} has some words in it.' for i in range(200)]
    big_html = (
        '<html><head><title>Long page</title><script>var tracking = 1;</script>'
        '<style>p { color: red; }</style></head>'
        '<body><nav><a href="/">Home</a></nav><article><h1>Long page</h1>'
        + ''.join(f'<p>{p}</p>' for p in paragraphs)
        + '</article></body></html>'
    )
    start = time.perf_counter()
    result = simple_json_from_html_string(big_html, use_readability=False)
    elapsed = time.perf_counter() - start
    texts = [block['text'] for block in result['plain_text']]
    print(f"Long page: {len(big_html)} characters of HTML -> {len(texts)} text blocks in {elapsed * 1000:.1f} ms")

    # Every paragraph survives, in order; script, style and navigation do not
    assert [t for t in texts if t.startswith('Paragraph number')] == paragraphs
    assert not any(s in result['content'] for s in ('tracking', 'color: red', 'Home')), result['content'][:200]
    assert result['title'] == 'Long page', result['title']

    # Extraction is deterministic, so results can be cached by a hash of the input
    assert simple_json_from_html_string(big_html, use_readability=False) == result

    # Malformed HTML is repaired by the parser instead of raising
    broken = simple_json_from_html_string('<p>Unclosed paragraph<p>Second <b>bold</div>', use_readability=False)
    print(f"Malformed input -> {broken['content']}")
    assert broken['content'] == '<div><p>Unclosed paragraph</p><p>Second bold</p></div>', broken['content']
    assert [block['text'] for block in broken['plain_text']] == ['Unclosed paragraph', 'Second bold'], broken

    # Empty, blank and script-only pages give an empty result rather than an error
    for empty in ('', '   ', '<html><body><script>x = 1</script></body></html>'):
        r = simple_json_from_html_string(empty, use_readability=False)
        print(f"  {empty!r:52} -> content={r['content']!r}, plain_text={r['plain_text']}")
        assert (r['title'], r['content'], r['plain_text']) == (None, '<div></div>', []), (empty, r)

    print("Large, malformed and empty input test completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"readabilipy example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e